# Senator Feature Base Functions

In [ ]:
# Senatör Featureunda yapmak istediğim:
# Senatörlerin rankingini sağlamak bu 2 farklı yolun birleşmesiyle olucak
# Biri senator trust index diyebiliriz bu geçmişe yönelik işlemlerde
# Bunda 2 spektrum yaparız buy farklı sell farklı olur 
# Transaction date ve received datedeki fiyatları karşılaştıracak
# Bu sayede aradaki sürede fiyat farkı az mı olmuş uzun vadeli mi
# Kısa vadeli mi yatırım yapıyor görüyor olacağız
# İkincisi senatör performans endeksi backtestle beraber
# Bir nevi senatörlerin performanslarını yorumlayacağız
# Bu elde ettiğimiz rankingi kullanacğımız bir faz olucak
# Ekonomik metriklerle sıraladığımız ve belli bir değerin üstünü yatırım
# için değerlendirmeye geçmeden yakın zamanda senatörler kendi
# rating katsayılarıyla beraber buy pozitif etki edicek
# sell negatif etki edicek sonrasında yine ranking hisselerimiz olacak

# Ayrıca yapılacaklar: 
# Otomatik power bi raporları
# JupyterHUBA geçiş ya da benzeri
# Trade box enjeksityonu
# Zipline a bak https://zipline.ml4trading.io



In [6]:
import pandas as pd
import numpy as np
import requests

API_KEY = open("FMP API KEY.txt").read().strip()
TICKER = "AAPL"  # örnek hisse

# === 1. Analyst Estimates verisi ===
url = f"https://financialmodelingprep.com/api/v3/analyst-estimates/{TICKER}?apikey={API_KEY}"
resp = requests.get(url)
data = resp.json()
df = pd.DataFrame(data)

# Tarihe göre sırala
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# === 2. Strateji: sadece estimatedEpsAvg'e göre basit sinyal ===
# Varsayım: tahmin ortalaması artıyorsa (önceki döneme göre) -> long
# düşüyorsa -> short
df['eps_change'] = df['estimatedEpsAvg'].diff()
df['signal'] = np.where(df['eps_change'] > 0, 1, -1)

# === 3. Getiri simülasyonu ===
# (dummy random return; fiyat endpoint bağlanınca gerçek getiri hesaplanabilir)
df['return'] = np.random.normal(0.001, 0.02, len(df))
df['strategy_return'] = df['signal'] * df['return']

# === 4. Performans metrikleri ===
sharpe_ratio = np.mean(df['strategy_return']) / np.std(df['strategy_return'], ddof=1) * np.sqrt(252)
hit_ratio = (df['strategy_return'] > 0).mean()
cumulative_return = (1 + df['strategy_return']).prod() - 1

cum_curve = (1 + df['strategy_return']).cumprod()
rolling_max = cum_curve.cummax()
drawdown = (cum_curve - rolling_max) / rolling_max
max_drawdown = drawdown.min()
calmar_ratio = cumulative_return / abs(max_drawdown)

print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Hit Ratio: {hit_ratio:.2%}")
print(f"Cumulative Return: {cumulative_return:.2%}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")


Sharpe Ratio: -0.75
Hit Ratio: 44.44%
Cumulative Return: -1.82%
Max Drawdown: -8.94%
Calmar Ratio: -0.20


In [7]:
import pandas as pd
import numpy as np
import requests

API_KEY = open("FMP API KEY.txt").read().strip()
TICKER = "AAPL"

# === 1. Analyst Estimates verisi ===
url = f"https://financialmodelingprep.com/api/v3/analyst-estimates/{TICKER}?apikey={API_KEY}"
resp = requests.get(url)
data = resp.json()
df = pd.DataFrame(data)

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# === 2. Strateji: sadece estimatedEpsAvg'e göre basit sinyal ===
df['eps_change'] = df['estimatedEpsAvg'].diff()
df['signal'] = np.where(df['eps_change'] > 0, 1, -1)

# === 3. Getiri simülasyonu (dummy) ===
df['return'] = np.random.normal(0.001, 0.02, len(df))
df['strategy_return'] = df['signal'] * df['return']

# === 4. Dönemsel metrik hesaplama (yıllık) ===
df.set_index('date', inplace=True)

def performance_metrics(x):
    sharpe = np.mean(x) / np.std(x, ddof=1) * np.sqrt(252) if np.std(x, ddof=1) > 0 else 0
    hit = (x > 0).mean()
    cum_return = (1 + x).prod() - 1
    cum_curve = (1 + x).cumprod()
    rolling_max = cum_curve.cummax()
    drawdown = (cum_curve - rolling_max) / rolling_max
    max_dd = drawdown.min() if not drawdown.empty else 0
    calmar = cum_return / abs(max_dd) if max_dd != 0 else np.nan
    return pd.Series({
        "Sharpe": sharpe,
        "Hit Ratio": hit,
        "Cumulative Return": cum_return,
        "Max Drawdown": max_dd,
        "Calmar": calmar
    })

# Yıllık metrikler
periodic_stats = df['strategy_return'].resample('Y').apply(performance_metrics)

print(periodic_stats)


Sharpe               0.000000
Hit Ratio            0.000000
Cumulative Return   -0.013589
Max Drawdown         0.000000
Calmar                    NaN
                       ...   
Sharpe               0.000000
Hit Ratio            1.000000
Cumulative Return    0.006326
Max Drawdown         0.000000
Calmar                    NaN
Name: strategy_return, Length: 170, dtype: float64


/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_16752/2505073299.py:46: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  periodic_stats = df['strategy_return'].resample('Y').apply(performance_metrics)


In [8]:
periodic_stats

Sharpe               0.000000
Hit Ratio            0.000000
Cumulative Return   -0.013589
Max Drawdown         0.000000
Calmar                    NaN
                       ...   
Sharpe               0.000000
Hit Ratio            1.000000
Cumulative Return    0.006326
Max Drawdown         0.000000
Calmar                    NaN
Name: strategy_return, Length: 170, dtype: float64

In [14]:
import requests
from bs4 import BeautifulSoup

url = "https://www.mediamarkt.com.tr/tr/category/lazer-epilasyonu-ve-aksesuarlari-582509.html?filter=brand:PHILIPS"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

# === Ürün isimlerini çek ===
names = [p.get_text(strip=True) for p in soup.find_all("p", {"data-test": "product-title"})]

# === Fiyatları çek ===
prices = [span.get_text(strip=True) for span in soup.find_all("span", {"aria-hidden": "true"})]

# Eşleştirme (isim + fiyat)
products = list(zip(names, prices))

# Yazdır
for name, price in products:
    print(name, ":", price)


PHILIPS SC 1997 Lumea Lazer IPL Epilasyon Cihazı : 
PHILIPS BRI953/01 Lumea IPL Tüy Alma Cihazı : 
PHILIPS BRI950/00 Lumea Yüz+Vücut+Hassas Bölge Kullanımı, Kablolu/Kablosuz Çanta Hediyeli IPL Lazer Epilasyon Tüy Alma Cihazı : 
PHILIPS BRI921/00 Lumea Yüz+Vücut+Hassas Bölge Kullanımı, Çanta ve Kaş Düzeltici Hediyeli IPL Lazer Epilasyon Tüy Alma Cihazı : 
PHILIPS Lumea BRI951/03 IPL Epilasyon Cihazı : 
PHILIPS BRI940/00 Lumea 8000 Series IPL Epilasyon Cihazı, Çanta Hediyeli, 1 Akıllı Başlık (Vücut) : 


In [16]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://www.mediamarkt.com.tr/tr/product/_philips-bri95000-lumea-yuzvucuthassas-bolge-kullanimi-kablolukablosuz-canta-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1180476.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

resp = requests.get(url, headers=headers)
html = resp.text

soup = BeautifulSoup(html, "html.parser")

# Fiyatı bulmak için span etiketini class ile yakalayalım
price_span = soup.find("span", {"class": "sc-5a9f6c31-0 ekmheE"})
if price_span:
    price = price_span.text.strip()
else:
    # Regex fallback (₺ ... rakamları yakala)
    match = re.search(r"₺\s*[\d\.\,]+", html)
    price = match.group(0) if match else "Bulunamadı"

print("Fiyat:", price)


Fiyat: ₺ 18.999,


In [18]:
import requests
from bs4 import BeautifulSoup

url = "https://www.mediamarkt.com.tr/tr/category/lazer-epilasyonu-ve-aksesuarlari-582509.html?filter=brand:PHILIPS"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

# Tüm <a> taglerini seç
links = [ "https://www.mediamarkt.com.tr" + a["href"] 
          for a in soup.select("a[data-test='mms-router-link-product-list-item-link']") 
          if a.get("href") ]

# Yazdır
for link in links:
    print(link)


https://www.mediamarkt.com.tr/tr/product/_philips-sc-1997-lumea-lazer-ipl-epilasyon-cihazi-1167383.html
https://www.mediamarkt.com.tr/tr/product/_philips-bri95301-lumea-ipl-tuy-alma-cihazi-1243309.html
https://www.mediamarkt.com.tr/tr/product/_philips-bri95000-lumea-yuzvucuthassas-bolge-kullanimi-kablolukablosuz-canta-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1180476.html
https://www.mediamarkt.com.tr/tr/product/_philips-bri92100-lumea-yuzvucuthassas-bolge-kullanimi-canta-ve-kas-duzeltici-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1183307.html
https://www.mediamarkt.com.tr/tr/product/_philips-lumea-bri95103-ipl-epilasyon-cihazi-1247595.html
https://www.mediamarkt.com.tr/tr/product/_philips-bri94000-lumea-8000-series-ipl-epilasyon-cihazi-canta-hediyeli-1-akilli-baslik-vucut-1232599.html


In [19]:
import requests
from bs4 import BeautifulSoup
import re
import time

base_url = "https://www.mediamarkt.com.tr"
category_url = "https://www.mediamarkt.com.tr/tr/category/lazer-epilasyonu-ve-aksesuarlari-582509.html?filter=brand:PHILIPS"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

# === 1. Kategori sayfasından ürün linklerini çek ===
resp = requests.get(category_url, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

links = [
    base_url + a["href"]
    for a in soup.select("a[data-test='mms-router-link-product-list-item-link']")
    if a.get("href")
]

# === 2. Her ürün sayfasına git ve fiyatını bul ===
products = []

for link in links:
    resp = requests.get(link, headers=headers)
    product_soup = BeautifulSoup(resp.text, "html.parser")

    # Ürün adı
    name_tag = product_soup.find("h1", {"data-test": "product-title"})
    name = name_tag.get_text(strip=True) if name_tag else "Ürün adı bulunamadı"

    # Fiyat
    price_tag = product_soup.find("span", {"class": "sc-5a9f6c31-0 ekmheE"})
    if price_tag:
        price = price_tag.text.strip()
    else:
        match = re.search(r"₺\s*[\d\.\,]+", resp.text)
        price = match.group(0) if match else "Fiyat bulunamadı"

    products.append({"name": name, "price": price, "link": link})
    print(name, ":", price, "|", link)

    time.sleep(1)  # siteyi yormamak için bekleme (1 sn)

Ürün adı bulunamadı : ₺ 9.699, | https://www.mediamarkt.com.tr/tr/product/_philips-sc-1997-lumea-lazer-ipl-epilasyon-cihazi-1167383.html
Ürün adı bulunamadı : ₺ 22.999, | https://www.mediamarkt.com.tr/tr/product/_philips-bri95301-lumea-ipl-tuy-alma-cihazi-1243309.html
Ürün adı bulunamadı : ₺ 18.999, | https://www.mediamarkt.com.tr/tr/product/_philips-bri95000-lumea-yuzvucuthassas-bolge-kullanimi-kablolukablosuz-canta-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1180476.html
Ürün adı bulunamadı : ₺ 12.699, | https://www.mediamarkt.com.tr/tr/product/_philips-bri92100-lumea-yuzvucuthassas-bolge-kullanimi-canta-ve-kas-duzeltici-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1183307.html
Ürün adı bulunamadı : ₺ 26.999, | https://www.mediamarkt.com.tr/tr/product/_philips-lumea-bri95103-ipl-epilasyon-cihazi-1247595.html
Ürün adı bulunamadı : ₺ 14.699, | https://www.mediamarkt.com.tr/tr/product/_philips-bri94000-lumea-8000-series-ipl-epilasyon-cihazi-canta-hediyeli-1-akilli-baslik-vucut-1232599.

In [20]:
import requests
from bs4 import BeautifulSoup
import re
import time

base_url = "https://www.mediamarkt.com.tr"
category_url = "https://www.mediamarkt.com.tr/tr/category/lazer-epilasyonu-ve-aksesuarlari-582509.html?filter=brand:PHILIPS"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

resp = requests.get(category_url, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

# === Ürün adları ve linkleri çek ===
product_cards = soup.select("a[data-test='mms-router-link-product-list-item-link']")

products = []

for card in product_cards:
    # ürün adı
    name_tag = card.find("p", {"data-test": "product-title"})
    name = name_tag.get_text(strip=True) if name_tag else "Ürün adı bulunamadı"

    # link
    link = base_url + card["href"]

    # her ürün sayfasına gir → fiyatı bul
    resp_detail = requests.get(link, headers=headers)
    detail_soup = BeautifulSoup(resp_detail.text, "html.parser")

    price_tag = detail_soup.find("span", {"class": "sc-5a9f6c31-0 ekmheE"})
    if price_tag:
        price = price_tag.get_text(strip=True)
    else:
        match = re.search(r"₺\s*[\d\.\,]+", resp_detail.text)
        price = match.group(0) if match else "Fiyat bulunamadı"

    products.append({"name": name, "price": price, "link": link})
    print(name, ":", price, "|", link)

    time.sleep(1)  # siteyi yormamak için bekleme


PHILIPS SC 1997 Lumea Lazer IPL Epilasyon Cihazı : ₺9.699, | https://www.mediamarkt.com.tr/tr/product/_philips-sc-1997-lumea-lazer-ipl-epilasyon-cihazi-1167383.html
PHILIPS BRI953/01 Lumea IPL Tüy Alma Cihazı : ₺22.999, | https://www.mediamarkt.com.tr/tr/product/_philips-bri95301-lumea-ipl-tuy-alma-cihazi-1243309.html
PHILIPS BRI950/00 Lumea Yüz+Vücut+Hassas Bölge Kullanımı, Kablolu/Kablosuz Çanta Hediyeli IPL Lazer Epilasyon Tüy Alma Cihazı : ₺18.999, | https://www.mediamarkt.com.tr/tr/product/_philips-bri95000-lumea-yuzvucuthassas-bolge-kullanimi-kablolukablosuz-canta-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1180476.html
PHILIPS BRI921/00 Lumea Yüz+Vücut+Hassas Bölge Kullanımı, Çanta ve Kaş Düzeltici Hediyeli IPL Lazer Epilasyon Tüy Alma Cihazı : ₺12.699, | https://www.mediamarkt.com.tr/tr/product/_philips-bri92100-lumea-yuzvucuthassas-bolge-kullanimi-canta-ve-kas-duzeltici-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1183307.html
PHILIPS Lumea BRI951/03 IPL Epilasyon Cihazı : ₺2

In [24]:
import requests
from bs4 import BeautifulSoup

url = "https://www.mediamarkt.com.tr/tr/brand/philips"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36"
}

# Sayfayı çek
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")


links = []
for a in soup.find_all("a", class_="sc-53d190fd-0 hUSSTO"):
    href = a.get("href")
    if href and href.startswith("https://www.mediamarkt.com.tr/tr/category"):
        links.append(href)

# Linkleri yazdır
for link in links:
    print(link)
    

https://www.mediamarkt.com.tr/tr/category/lazer-epilasyonu-ve-aksesuarlari-582509.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/erkek-bakim-urunleri-465821.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/epilasyon-465827.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/sac-bakim-675547.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/agiz-bakim-urunleri-465839.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/espresso-kahve-makineleri-806540.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/filtre-kahve-makineleri-806538.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/turk-kahve-makineleri-cezveler-806539.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/hava-nemlendirici-temizleyiciler-465759.html?filter=brand:PHILIPS
https://www.mediamarkt.com.tr/tr/category/buhar-kazanli-utu-465749.html?filter=brand:PHILIPS
https://www.mediamarkt.c

In [25]:
for i in links:
    category_url = i
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    }

    resp = requests.get(category_url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    # === Ürün adları ve linkleri çek ===
    product_cards = soup.select("a[data-test='mms-router-link-product-list-item-link']")

    products = []

    for card in product_cards:
        # ürün adı
        name_tag = card.find("p", {"data-test": "product-title"})
        name = name_tag.get_text(strip=True) if name_tag else "Ürün adı bulunamadı"

        # link
        link = base_url + card["href"]

        # her ürün sayfasına gir → fiyatı bul
        resp_detail = requests.get(link, headers=headers)
        detail_soup = BeautifulSoup(resp_detail.text, "html.parser")

        price_tag = detail_soup.find("span", {"class": "sc-5a9f6c31-0 ekmheE"})
        if price_tag:
            price = price_tag.get_text(strip=True)
        else:
            match = re.search(r"₺\s*[\d\.\,]+", resp_detail.text)
            price = match.group(0) if match else "Fiyat bulunamadı"

        products.append({"name": name, "price": price, "link": link})
        print(name, ":", price, "|", link)

        time.sleep(1)  # siteyi yormamak için bekleme


PHILIPS SC 1997 Lumea Lazer IPL Epilasyon Cihazı : ₺9.699, | https://www.mediamarkt.com.tr/tr/product/_philips-sc-1997-lumea-lazer-ipl-epilasyon-cihazi-1167383.html
PHILIPS BRI953/01 Lumea IPL Tüy Alma Cihazı : ₺22.999, | https://www.mediamarkt.com.tr/tr/product/_philips-bri95301-lumea-ipl-tuy-alma-cihazi-1243309.html
PHILIPS BRI950/00 Lumea Yüz+Vücut+Hassas Bölge Kullanımı, Kablolu/Kablosuz Çanta Hediyeli IPL Lazer Epilasyon Tüy Alma Cihazı : ₺18.999, | https://www.mediamarkt.com.tr/tr/product/_philips-bri95000-lumea-yuzvucuthassas-bolge-kullanimi-kablolukablosuz-canta-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1180476.html
PHILIPS BRI921/00 Lumea Yüz+Vücut+Hassas Bölge Kullanımı, Çanta ve Kaş Düzeltici Hediyeli IPL Lazer Epilasyon Tüy Alma Cihazı : ₺12.699, | https://www.mediamarkt.com.tr/tr/product/_philips-bri92100-lumea-yuzvucuthassas-bolge-kullanimi-canta-ve-kas-duzeltici-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1183307.html
PHILIPS Lumea BRI951/03 IPL Epilasyon Cihazı : ₺2

In [36]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

base_url = "https://www.mediamarkt.com.tr"

# Örnek kategori linkleri (sen bunları links listende topluyorsun zaten)
links = [
    "https://www.mediamarkt.com.tr/tr/category/lazer-epilasyonu-ve-aksesuarlari-582509.html?filter=brand:PHILIPS"
    # buraya diğer kategori linklerini de ekleyebilirsin
]

all_products = []  # tüm kategorilerdeki ürünler buraya

for i in links:
    category_url = i
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    }

    resp = requests.get(category_url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    # === Ürün adları ve linkleri çek ===
    product_cards = soup.select("a[data-test='mms-router-link-product-list-item-link']")

    for card in product_cards:
        # ürün adı
        name_tag = card.find("p", {"data-test": "product-title"})
        name = name_tag.get_text(strip=True) if name_tag else "Ürün adı bulunamadı"

        # link
        link = base_url + card["href"]

        # her ürün sayfasına gir → fiyatı bul
        resp_detail = requests.get(link, headers=headers)
        detail_soup = BeautifulSoup(resp_detail.text, "html.parser")

        price_tag = detail_soup.find("span", {"class": "sc-5a9f6c31-0 ekmheE"})
        if price_tag:
            price = price_tag.get_text(strip=True)
        else:
            match = re.search(r"₺\s*[\d\.\,]+", resp_detail.text)
            price = match.group(0) if match else "Fiyat bulunamadı"

        all_products.append({"Kategori": category_url, "Ürün Adı": name, "Fiyat": price, "Link": link})
        print(name, ":", price, "|", link)

        time.sleep(1)  # siteyi yormamak için bekleme

# === DataFrame oluştur ve Excel'e kaydet ===
df = pd.DataFrame(all_products)
df.to_excel("mediamarkt_philips.xlsx", index=False)

print("✅ Veriler 'mediamarkt_philips.xlsx' dosyasına kaydedildi.")


PHILIPS SC 1997 Lumea Lazer IPL Epilasyon Cihazı : ₺9.699, | https://www.mediamarkt.com.tr/tr/product/_philips-sc-1997-lumea-lazer-ipl-epilasyon-cihazi-1167383.html
PHILIPS BRI953/01 Lumea IPL Tüy Alma Cihazı : ₺22.999, | https://www.mediamarkt.com.tr/tr/product/_philips-bri95301-lumea-ipl-tuy-alma-cihazi-1243309.html
PHILIPS BRI950/00 Lumea Yüz+Vücut+Hassas Bölge Kullanımı, Kablolu/Kablosuz Çanta Hediyeli IPL Lazer Epilasyon Tüy Alma Cihazı : ₺18.999, | https://www.mediamarkt.com.tr/tr/product/_philips-bri95000-lumea-yuzvucuthassas-bolge-kullanimi-kablolukablosuz-canta-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1180476.html
PHILIPS BRI921/00 Lumea Yüz+Vücut+Hassas Bölge Kullanımı, Çanta ve Kaş Düzeltici Hediyeli IPL Lazer Epilasyon Tüy Alma Cihazı : ₺12.699, | https://www.mediamarkt.com.tr/tr/product/_philips-bri92100-lumea-yuzvucuthassas-bolge-kullanimi-canta-ve-kas-duzeltici-hediyeli-ipl-lazer-epilasyon-tuy-alma-cihazi-1183307.html
PHILIPS Lumea BRI951/03 IPL Epilasyon Cihazı : ₺2

In [39]:
import requests
from bs4 import BeautifulSoup
import re
import time

category_url = "https://www.trendyol.com/sr?q=PHILIPS%20SC%201997%20Lumea%20Lazer%20IPL%20Epilasyon%20Cihazı&qt=PHILIPS%20SC%201997%20Lumea%20Lazer%20IPL%20Epilasyon%20Cihazı&st=PHILIPS%20SC%201997%20Lumea%20Lazer%20IPL%20Epilasyon%20Cihazı&os=1"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

resp = requests.get(category_url, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

soup

<!DOCTYPE html>

<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>
<title>Attention Required! | Cloudflare</title>
<meta charset="utf-8"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
<meta content="noindex, nofollow" name="robots"/>
<meta content="width=device-width,initial-scale=1" name="viewport"/>
<link href="/cdn-cgi/styles/cf.errors.css" id="cf_styles-css" rel="stylesheet"/>
<!--[if lt IE 9]><link rel="stylesheet" id='cf_styles-ie-css' href="/cdn-cgi/styles/cf.errors.ie.css" /><![endif]-->
<style>body{margin:0;padding:0}</style>
<!--[if gte IE 10]><!-->
<script>
  if (!navigator.cookieEnabled) {
    window.addEventListener('DOMContentLoaded', function

In [40]:
url = 'https://www.trendyol.com/nike/air-force-1-low-valentine-s-day-2023-p-838721121?boutiqueId=61&merchantId=763411'
sayfa = requests.get(url)
html_sayfa = BeautifulSoup(page)

#while True:
url = "https://www.trendyol.com/nike/academy-team-s-futbol-spor-cantasi-cu8097-010-p-101456942"
sayfa = requests.get(url)
html_sayfa = BeautifulSoup(sayfa.content,"html.parser")
isim = html_sayfa.find("h1",class_="pr-new-br").getText()
print(isim)
fiyat = html_sayfa.find("span",class_="prc-dsc").getText()
print(fiyat)

NameError: name 'page' is not defined